In [ ]:
############################################################
# In silico perturbation analysis to evaluate the regulatory
# effect of the lincRNA locus on the target gene using
# AlphaGenome
#
# Author: Shiyu Zhang (Shelley)
# Last Date: 2026-02-27
############################################################

In [ ]:
# step.1 setup and imports
from alphagenome import colab_utils
from alphagenome.data import gene_annotation, genome, track_data, transcript
from alphagenome.models import dna_client
from alphagenome.visualization import plot_components
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

dna_model = dna_client.create(api_key="YOUR_API_KEY_here")

# Load metadata objects for human.
output_metadata = dna_model.output_metadata(
    organism=dna_client.Organism.HOMO_SAPIENS
)

# Load gene annotations (from GENCODE).
gtf = pd.read_feather(
    'https://storage.googleapis.com/alphagenome/reference/gencode/'
    'hg38/gencode.v46.annotation.gtf.gz.feather'
)

E0227 20:09:41.257198 61559650 ssl_transport_security_utils.cc:114] Corruption detected.
E0227 20:09:41.257234 61559650 ssl_transport_security_utils.cc:71] error:1e000065:Cipher functions:OPENSSL_internal:BAD_DECRYPT
E0227 20:09:41.257251 61559650 ssl_transport_security_utils.cc:71] error:1000008b:SSL routines:OPENSSL_internal:DECRYPTION_FAILED_OR_BAD_RECORD_MAC
E0227 20:09:41.257255 61559650 secure_endpoint.cc:254] Decryption error: TSI_DATA_CORRUPTED


In [3]:
############################################################
# Step 2: Define prediction regions (hg38)
############################################################

# Input coordinates (as typically reported: 1-based, inclusive)
lincRNA_chr = "chr8"
lincRNA_start_1based = 15_973_315
lincRNA_end_1based   = 16_000_324

tusc3_chr = "chr8"
tusc3_start_1based = 15_417_188
tusc3_end_1based   = 15_852_091

# Convert to AlphaGenome Interval coordinates (0-based start, end-exclusive)
lincRNA_interval = genome.Interval(
    chromosome=lincRNA_chr,
    start=lincRNA_start_1based - 1,
    end=lincRNA_end_1based,  # inclusive -> exclusive
    strand="+",
    name="ENSG00000253648"
)

tusc3_interval = genome.Interval(
    chromosome=tusc3_chr,
    start=tusc3_start_1based - 1,
    end=tusc3_end_1based,
    strand="+",
    name="TUSC3"
)

# Span that covers both loci (minimal joint span)
joint_span = genome.Interval(
    chromosome="chr8",
    start=min(lincRNA_interval.start, tusc3_interval.start),
    end=max(lincRNA_interval.end, tusc3_interval.end),
    strand=".",
    name="ENSG00000253648_to_TUSC3_span"
)

# Resize to AlphaGenome required input length (1 Mb)
interval = joint_span.resize(dna_client.SEQUENCE_LENGTH_1MB)

joint_span, interval
# sanity check: print the width for these regions
print("lincRNA_interval width (bp):", lincRNA_interval.width)
print("tusc3_interval width (bp):", tusc3_interval.width)
print("joint_span width (bp):", joint_span.width)
print("interval (1Mb) width (bp):", interval.width)

lincRNA_interval width (bp): 27010
tusc3_interval width (bp): 434904
joint_span width (bp): 583137
interval (1Mb) width (bp): 1048576


In [4]:
############################################################
# Step 3 (Random-replacement-null): Build 101 sequences
#   - 1 reference sequence (1 Mb interval)
#   - 100 sequences where ONLY the lincRNA body is replaced
#     by a random A/C/G/T string of the same length
############################################################

from pyfaidx import Fasta
import numpy as np
import gc

HG38_FA = "/Users/shelleyz/Projects/Melanoma-WGS/ncRNA.AlphaGenome/hg38.fa"

# --- Load FASTA ---
fa = Fasta(HG38_FA, as_raw=True, sequence_always_upper=True)

def fetch_seq(fa: Fasta, interval: genome.Interval) -> str:
    chrom = getattr(interval, "chromosome", None) or getattr(interval, "chrom", None)
    if chrom is None:
        raise AttributeError("Interval has no attribute 'chromosome' (or 'chrom').")
    return str(fa[chrom][interval.start:interval.end])

# 1) extract reference 1Mb sequence
ref_seq_1mb = fetch_seq(fa, interval)

# --- Release FASTA from memory ---
fa.close()
del fa
gc.collect()

# 2) locate lincRNA body within 1Mb interval
linc_chr = getattr(lincRNA_interval, "chromosome", None) or getattr(lincRNA_interval, "chrom", None)
int_chr  = getattr(interval, "chromosome", None) or getattr(interval, "chrom", None)

if linc_chr != int_chr:
    raise ValueError("lincRNA_interval and interval must be on the same chromosome.")

rel_start = lincRNA_interval.start - interval.start
rel_end   = lincRNA_interval.end   - interval.start

if not (0 <= rel_start < rel_end <= len(ref_seq_1mb)):
    raise ValueError("lincRNA_interval must be fully contained within the 1 Mb interval.")

linc_ref_seq = ref_seq_1mb[rel_start:rel_end]
L = len(linc_ref_seq)

if L <= 0:
    raise ValueError("Computed lincRNA length <= 0 (check interval coordinates).")

# 3) generate 100 random-replacement sequences
N_RANDOM = 100
rng = np.random.default_rng(20260227)
alphabet = np.array(list("ACGT"), dtype="S1")

random_seqs_1mb = []
for _ in range(N_RANDOM):
    rand_linc = alphabet[rng.integers(0, 4, size=L)].tobytes().decode("ascii")
    random_seqs_1mb.append(
        ref_seq_1mb[:rel_start] + rand_linc + ref_seq_1mb[rel_end:]
    )

# 4) final list: 1 reference + 100 random-replacement backgrounds
all_seqs = [ref_seq_1mb] + random_seqs_1mb

# --- Sanity checks ---
assert len(all_seqs) == 101
assert all(len(s) == len(ref_seq_1mb) for s in all_seqs)
assert all_seqs[0][rel_start:rel_end] != all_seqs[1][rel_start:rel_end]
assert all_seqs[0][:rel_start] == all_seqs[1][:rel_start]
assert all_seqs[0][rel_end:]  == all_seqs[1][rel_end:]

print("Total sequences:", len(all_seqs))
print("1 Mb interval length:", len(ref_seq_1mb))
print("lincRNA length:", L)
print("Sanity check passed.")

Total sequences: 101
1 Mb interval length: 1048576
lincRNA length: 27010
Sanity check passed.


In [5]:
# Print reference 1 Mb sequence (truncated for readability)
print("\n=== Reference 1Mb sequence (first 200 bp) ===")
print(ref_seq_1mb[:200])

# Print one random-replacement sequence (truncated)
print("\n=== Random-replacement 1Mb sequence (first 200 bp) ===")
print(all_seqs[1][:200])

# Also explicitly print the lincRNA segment only (first 120 bp) for clarity
print("\n=== Reference lincRNA segment (first 120 bp) ===")
print(all_seqs[0][rel_start:rel_start+120])

print("\n=== Random lincRNA segment (first 120 bp) ===")
print(all_seqs[1][rel_start:rel_start+120])


=== Reference 1Mb sequence (first 200 bp) ===
TGAGTTTCCAGTTACTCTTCACCTCTCATGCATTTTCTCCTTTTTTTTTTCTTTTGTCCTACACCAGGCCTCGGGAGACTGGTCCCGAAGGATTTCATCACCCTAGATCTCTTGCACTCTGGCCTCACTAGGTTCAGGCAACGGGAGTCCCCAGCAGGAAACTGAGAGGTGAGGGAAGAGAGAGCAGGCGATTTCTTCAC

=== Random-replacement 1Mb sequence (first 200 bp) ===
TGAGTTTCCAGTTACTCTTCACCTCTCATGCATTTTCTCCTTTTTTTTTTCTTTTGTCCTACACCAGGCCTCGGGAGACTGGTCCCGAAGGATTTCATCACCCTAGATCTCTTGCACTCTGGCCTCACTAGGTTCAGGCAACGGGAGTCCCCAGCAGGAAACTGAGAGGTGAGGGAAGAGAGAGCAGGCGATTTCTTCAC

=== Reference lincRNA segment (first 120 bp) ===
CTCTCACAGGAATCTTGTTTGTACCAGCAGATGCCCTTGCGGCTCTTATCTGATCTATGTCCACTTTATTCCTGTCATTATAACCACTCTCTAAGAGAGCTGTGAATAGAAATAAGGTGA

=== Random lincRNA segment (first 120 bp) ===
ACCATTCAACTGATTTTAATCCTCAATGTACTGCAGTCCTTACGACTACTCACTATGACACAATTACTAAACCCTATAACCCCTCTACAACCGACGGGTCTTGCGTGTGGTGAGGTACGA


In [6]:
############################################################
# Step 4 (updated)
#   - Keep strand == "+"
#   - Keep biosamples that support BOTH RNA_SEQ and CAGE
############################################################

from alphagenome.models import dna_client
import pandas as pd

md = dna_model.output_metadata(dna_client.Organism.HOMO_SAPIENS).concatenate()
md = md.copy()

# Print md structure (easy to screenshot)
print("shape:", md.shape)
print("\ncolumns:")
print(list(md.columns))
print("\ndtypes:")
print(md.dtypes)
print("\nhead:")
print(md.head(5))

# Convert OutputType enum to clean string
md["output_type_str"] = md["output_type"].astype(str).str.replace("OutputType.", "", regex=False)

# ---------------------------------------------------------
# 1️⃣ Keep strand == "+"
# ---------------------------------------------------------
md = md[md["strand"] == "+"].copy()
print("\nAfter strand filter (+):", md.shape)

# ---------------------------------------------------------
# 2️⃣ Keep biosamples that contain BOTH RNA_SEQ and CAGE
# ---------------------------------------------------------

# Group biosample -> set of available output types
biosample_to_outputs = (
    md.groupby("biosample_name")["output_type_str"]
      .apply(set)
)

# Select biosamples that contain both RNA_SEQ and CAGE
valid_biosamples = [
    b for b, outputs in biosample_to_outputs.items()
    if {"RNA_SEQ", "CAGE"}.issubset(outputs)
]

print("\nBiosamples with BOTH RNA_SEQ and CAGE:", len(valid_biosamples))

# Filter md to only those biosamples
md_filtered = md[md["biosample_name"].isin(valid_biosamples)].copy()
print("Filtered md shape:", md_filtered.shape)

# ---------------------------------------------------------
# Extract ontology terms for prediction
# ---------------------------------------------------------

ontology_terms = sorted(
    md_filtered["ontology_curie"]
        .dropna()
        .astype(str)
        .unique()
)

print("\nFinal ontology_terms count:", len(ontology_terms))
print("Example ontology_terms (first 20):")
print(ontology_terms[:20])

############################################################
# Sanity check:
# For each filtered biosample, count RNA_SEQ and CAGE tracks
############################################################

# Count tracks per biosample per output type
track_counts = (
    md_filtered.groupby(["biosample_name", "output_type_str"])
      .size()
      .unstack(fill_value=0)
)

print("\nTrack counts per biosample (RNA_SEQ / CAGE):")
print(track_counts.head(20))

print("\nSummary statistics:")
print(track_counts.describe())

# Ensure each biosample has at least 1 RNA_SEQ and 1 CAGE
assert (track_counts["RNA_SEQ"] > 0).all(), "Some biosamples missing RNA_SEQ"
assert (track_counts["CAGE"] > 0).all(), "Some biosamples missing CAGE"

print("\nSanity check passed: Every biosample has both RNA_SEQ and CAGE (+ strand).")
print("Total filtered biosamples:", track_counts.shape[0])

shape: (5563, 15)

columns:
['name', 'strand', 'Assay title', 'ontology_curie', 'biosample_name', 'biosample_type', 'biosample_life_stage', 'data_source', 'endedness', 'genetically_modified', 'nonzero_mean', 'output_type', 'gtex_tissue', 'histone_mark', 'transcription_factor']

dtypes:
name                     object
strand                   object
Assay title              object
ontology_curie           object
biosample_name           object
biosample_type           object
biosample_life_stage     object
data_source              object
endedness                object
genetically_modified     object
nonzero_mean            float64
output_type              object
gtex_tissue              object
histone_mark             object
transcription_factor     object
dtype: object

head:
                  name strand Assay title ontology_curie  \
0  CL:0000084 ATAC-seq      .    ATAC-seq     CL:0000084   
1  CL:0000100 ATAC-seq      .    ATAC-seq     CL:0000100   
2  CL:0000236 ATAC-seq      .   

In [7]:
############################################################
# Step 5A — Predict reference sequence only
############################################################

import time
import pickle
from alphagenome.models import dna_client

REQUESTED_OUTPUTS = {
    dna_client.OutputType.RNA_SEQ,
    dna_client.OutputType.CAGE,
}

ref_seq = all_seqs[0]

# sanity check
assert len(ref_seq) == interval.width, (len(ref_seq), interval.width)

print("Running reference prediction only...")
print("Requested outputs:", [str(x) for x in REQUESTED_OUTPUTS])
print("Number of ontology_terms:", len(ontology_terms))

t0 = time.time()

out_ref = dna_model.predict_sequence(
    sequence=ref_seq,
    interval=interval,
    requested_outputs=REQUESTED_OUTPUTS,
    ontology_terms=ontology_terms,
)

print(f"Reference finished in {time.time() - t0:.2f}s")
print("Output type:", type(out_ref))

Running reference prediction only...
Requested outputs: ['OutputType.RNA_SEQ', 'OutputType.CAGE']
Number of ontology_terms: 83
Reference finished in 150.43s
Output type: <class 'alphagenome.models.dna_output.Output'>


In [8]:
############################################################
# Save reference prediction output (uncompressed)
############################################################

import os
import pickle

SAVE_DIR = "/Users/shelleyz/Projects/Melanoma-WGS/ncRNA.AlphaGenome/Predictions"
os.makedirs(SAVE_DIR, exist_ok=True)

OUT_REF_PATH = os.path.join(SAVE_DIR, "alphagenome_out_ref.pkl")

with open(OUT_REF_PATH, "wb") as f:
    pickle.dump(out_ref, f, protocol=pickle.HIGHEST_PROTOCOL)

print("Saved reference output to:")
print(OUT_REF_PATH)

# Optional: show file size
size_mb = os.path.getsize(OUT_REF_PATH) / (1024**2)
print(f"File size: {size_mb:.2f} MB")

Saved reference output to:
/Users/shelleyz/Projects/Melanoma-WGS/ncRNA.AlphaGenome/Predictions/alphagenome_out_ref.pkl
File size: 1636.05 MB


E0227 20:14:51.438648 61559650 ssl_transport_security_utils.cc:114] Corruption detected.
E0227 20:14:51.438813 61559650 ssl_transport_security_utils.cc:71] error:1e000065:Cipher functions:OPENSSL_internal:BAD_DECRYPT
E0227 20:14:51.438823 61559650 ssl_transport_security_utils.cc:71] error:1000008b:SSL routines:OPENSSL_internal:DECRYPTION_FAILED_OR_BAD_RECORD_MAC
E0227 20:14:51.438827 61559650 secure_endpoint.cc:254] Decryption error: TSI_DATA_CORRUPTED


In [9]:
############################################################
# Step 5B (test): Predict RNA-seq + CAGE for first 10 random sequences
#   - sequential loop (easy to debug)
#   - prints progress
#   - saves each output to disk for later loading
############################################################

import os
import time
import pickle
from alphagenome.models import dna_client

# --- inputs assumed from earlier steps ---
# interval        : 1Mb genome.Interval
# all_seqs        : list[str] (all_seqs[0] is reference; all_seqs[1:] are randomized)
# ontology_terms  : list[str]
# dna_model       : AlphaGenome DNA model client

REQUESTED_OUTPUTS = {
    dna_client.OutputType.RNA_SEQ,
    dna_client.OutputType.CAGE,
}

SAVE_DIR = "/Users/shelleyz/Projects/Melanoma-WGS/ncRNA.AlphaGenome/Predictions"
OUTDIR = os.path.join(SAVE_DIR, "random10")
os.makedirs(OUTDIR, exist_ok=True)

rand_seqs_10 = all_seqs[1:11]  # first 10 randomized sequences
assert len(rand_seqs_10) == 10
assert all(len(s) == interval.width for s in rand_seqs_10)

def predict_one(seq: str):
    return dna_model.predict_sequence(
        sequence=seq,
        interval=interval,
        requested_outputs=REQUESTED_OUTPUTS,
        ontology_terms=ontology_terms,
    )

print(f"Running predictions for {len(rand_seqs_10)} randomized sequences...")
print("Saving outputs to:", OUTDIR)

t_all = time.time()

for i, seq in enumerate(rand_seqs_10, start=1):
    out_path = os.path.join(OUTDIR, f"alphagenome_out_rand_{i:02d}.pkl")

    # skip if already computed
    if os.path.exists(out_path):
        size_mb = os.path.getsize(out_path) / (1024**2)
        print(f"[{i:02d}/10] exists -> skip ({size_mb:.2f} MB): {os.path.basename(out_path)}")
        continue

    t0 = time.time()
    print(f"[{i:02d}/10] predicting ...", end="", flush=True)

    out_i = predict_one(seq)

    dt = time.time() - t0
    with open(out_path, "wb") as f:
        pickle.dump(out_i, f, protocol=pickle.HIGHEST_PROTOCOL)

    size_mb = os.path.getsize(out_path) / (1024**2)
    elapsed = time.time() - t_all
    rate = i / elapsed if elapsed > 0 else float("nan")
    remaining = (10 - i) / rate if rate and rate > 0 else float("nan")

    print(f" done in {dt:.1f}s | saved {size_mb:.2f} MB | "
          f"elapsed {elapsed/60:.1f} min | ~{remaining/60:.1f} min left")

print("Finished random10 predictions.")

Running predictions for 10 randomized sequences...
Saving outputs to: /Users/shelleyz/Projects/Melanoma-WGS/ncRNA.AlphaGenome/Predictions/random10
[01/10] predicting ... done in 151.5s | saved 1636.05 MB | elapsed 2.5 min | ~22.9 min left
[02/10] predicting ... done in 151.8s | saved 1636.05 MB | elapsed 5.1 min | ~20.4 min left
[03/10] predicting ... done in 150.6s | saved 1636.05 MB | elapsed 7.6 min | ~17.8 min left
[04/10] predicting ... done in 150.9s | saved 1636.05 MB | elapsed 10.1 min | ~15.2 min left
[05/10] predicting ... done in 159.9s | saved 1636.05 MB | elapsed 12.8 min | ~12.8 min left
[06/10] predicting ... done in 164.5s | saved 1636.05 MB | elapsed 15.6 min | ~10.4 min left
[07/10] predicting ... done in 172.0s | saved 1636.05 MB | elapsed 18.5 min | ~7.9 min left
[08/10] predicting ... done in 227.7s | saved 1636.05 MB | elapsed 22.3 min | ~5.6 min left
[09/10] predicting ... done in 237.8s | saved 1636.05 MB | elapsed 26.3 min | ~2.9 min left
[10/10] predicting ...